# 02 — Test Benchmark (Cross-Polytope)

Loads calibrated parameters from `01_calibration` and runs the test
benchmark for the **linear baseline** and **CrossPolytope** across
the 3 hash seeds. Test queries only — no parameter selection here.


In [ ]:
import numpy as np
import pandas as pd
import time
import json
import pathlib
import sys

try:
    from tqdm.notebook import tqdm
except ImportError:
    import tqdm

root_dir = pathlib.Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))


In [ ]:
# ============================================================
# CONFIG
# ============================================================
SPLIT_SEED = 42
DATASET = 'heart'
BASE_SIZE = None

HEART_CSV_PATH = ''
CHEST_EMBEDDINGS_PATH = ''
CHEST_LABELS_PATH = ''
CALIB_DIR = 'results'   # contains best_params_seed{SPLIT_SEED}.json
# ============================================================


In [ ]:
from config.seeds import SPLIT_SEEDS, HASH_SEEDS
from config.protocol import TIMING_WARMUP, TIMING_REPEATS
from config.metrics import K_NEIGHBOURS
assert SPLIT_SEED in SPLIT_SEEDS


In [ ]:
with open(pathlib.Path(CALIB_DIR) / f'best_params_seed{SPLIT_SEED}.json') as f:
    best_params = json.load(f)
cp_params = best_params['crosspolytope']
print('Calibrated CrossPolytope params:', cp_params)


In [ ]:
if DATASET == 'heart':
    from src.datasets.heart_disease import load_and_split
    X_base, y_base, X_val, y_val, X_test, y_test = load_and_split(
        HEART_CSV_PATH, split_seed=SPLIT_SEED, normalise_cosine=True
    )
else:
    from src.datasets.chest_xray14 import load_and_split
    X_base, y_base, X_val, y_val, X_test, y_test = load_and_split(
        CHEST_EMBEDDINGS_PATH, CHEST_LABELS_PATH,
        split_seed=SPLIT_SEED, normalise_cosine=True,
    )
    if BASE_SIZE is not None:
        idx = np.random.default_rng(SPLIT_SEED).choice(
            len(X_base), size=min(BASE_SIZE, len(X_base)), replace=False
        )
        X_base = X_base[idx]
        y_base = y_base[idx]

print(f'X_base: {X_base.shape}, X_test: {X_test.shape}')


In [ ]:
from src.indexes.linear_exact import LinearExact
from src.indexes.cross_polytope import CrossPolytope
from src.evaluation.metrics import (
    recall_at_k, ndcg_at_k, candidate_fraction,
    macro_f1, balanced_accuracy,
)

def majority_vote(labels):
    if len(labels) == 0:
        return -1
    values, counts = np.unique(labels, return_counts=True)
    return values[np.argmax(counts)]


In [ ]:
gt_path = pathlib.Path('results') / f'linear_gt_{DATASET}_{BASE_SIZE}_seed{SPLIT_SEED}.npz'
lin_idx = LinearExact()
lin_idx.build(X_base)

gt_indices, gt_distances = [], []
lin_lat, lin_preds = [], []
for q in X_test:
    t0 = time.perf_counter()
    for _ in range(TIMING_WARMUP):
        lin_idx.query(q, k=K_NEIGHBOURS)
    times = []
    for _ in range(TIMING_REPEATS):
        t1 = time.perf_counter()
        idxs, dists = lin_idx.query(q, k=K_NEIGHBOURS)
        times.append(time.perf_counter() - t1)
    lin_lat.append(float(np.median(times)))
    gt_indices.append(idxs)
    gt_distances.append(dists)
    lin_preds.append(majority_vote(y_base[idxs]))

np.savez(gt_path, indices=np.array(gt_indices), distances=np.array(gt_distances))
print(f'Linear GT + timing done: median {np.median(lin_lat)*1e3:.2f} ms')


In [ ]:
quality_records, timing_records = [], []
candidate_records, memory_records = [], []

def add_linear_records():
    quality_records.append({
        'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'hash_seed': 'exact', 'method': 'linear',
        'recall@1': 1.0, 'recall@5': 1.0, 'recall@10': 1.0, 'ndcg@10': 1.0,
        'query_coverage': 1.0, 'empty_rate': 0.0,
        'accuracy': float(np.mean(y_test == lin_preds)),
        'macro_f1': macro_f1(y_test, lin_preds),
        'balanced_accuracy': balanced_accuracy(y_test, lin_preds),
    })
    timing_records.append({
        'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'hash_seed': 'exact', 'method': 'linear',
        'median_latency_s': float(np.median(lin_lat)),
        'mean_latency_s': float(np.mean(lin_lat)),
        'p95_latency_s': float(np.percentile(lin_lat, 95)),
        'build_time_s': lin_idx.build_time_s,
    })
    candidate_records.append({
        'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'hash_seed': 'exact', 'method': 'linear',
        'mean_candidates': float(len(X_base)), 'mean_cand_fraction': 1.0, 'mean_buckets_visited': 1.0,
    })
    memory_records.append({
        'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'hash_seed': 'exact', 'method': 'linear', 'index_memory_bytes': X_base.nbytes,
    })

add_linear_records()


In [ ]:
for h_seed in tqdm(HASH_SEEDS, desc='CrossPolytope hash seeds'):
    cp = CrossPolytope(
        num_hashes=cp_params['num_hashes'],
        num_tables=cp_params['num_tables'],
        num_rotations=cp_params['num_rotations'],
        num_probes=cp_params['num_probes'],
        hash_seed=h_seed,
    )
    cp.build(X_base)

    lat, r1, r5, r10, ndcg, cand, bkt, cov, preds = [], [], [], [], [], [], [], [], []
    for i, q in enumerate(X_test):
        t0 = time.perf_counter()
        idxs, _, n_cand, n_b = cp.query(q, k=K_NEIGHBOURS)
        lat.append(time.perf_counter() - t0)
        r1.append(recall_at_k(idxs, gt_indices[i], 1))
        r5.append(recall_at_k(idxs, gt_indices[i], 5))
        r10.append(recall_at_k(idxs, gt_indices[i], 10))
        ndcg.append(ndcg_at_k(idxs, gt_indices[i], 10))
        cand.append(n_cand)
        bkt.append(n_b)
        cov.append(float(len(idxs) >= K_NEIGHBOURS))
        preds.append(majority_vote(y_base[idxs]))

    quality_records.append({
        'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'hash_seed': h_seed, 'method': 'crosspolytope',
        'recall@1': float(np.mean(r1)), 'recall@5': float(np.mean(r5)),
        'recall@10': float(np.mean(r10)), 'ndcg@10': float(np.mean(ndcg)),
        'query_coverage': float(np.mean(cov)), 'empty_rate': float(np.mean(np.array(cand) == 0)),
        'accuracy': float(np.mean(y_test == preds)),
        'macro_f1': macro_f1(y_test, preds),
        'balanced_accuracy': balanced_accuracy(y_test, preds),
    })
    timing_records.append({
        'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'hash_seed': h_seed, 'method': 'crosspolytope',
        'median_latency_s': float(np.median(lat)),
        'mean_latency_s': float(np.mean(lat)),
        'p95_latency_s': float(np.percentile(lat, 95)),
        'build_time_s': cp.build_time_s,
    })
    candidate_records.append({
        'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'hash_seed': h_seed, 'method': 'crosspolytope',
        'mean_candidates': float(np.mean(cand)),
        'mean_cand_fraction': float(np.mean(cand)) / len(X_base),
        'mean_buckets_visited': float(np.mean(bkt)),
    })
    memory_records.append({
        'dataset': DATASET, 'base_size': str(BASE_SIZE), 'split_seed': SPLIT_SEED,
        'hash_seed': h_seed, 'method': 'crosspolytope',
        'index_memory_bytes': cp.estimate_memory_bytes(),
    })


In [ ]:
OUTPUT_DIR = pathlib.Path('results')
OUTPUT_DIR.mkdir(exist_ok=True)
tag = f'{DATASET}_{BASE_SIZE}_seed{SPLIT_SEED}'

pd.DataFrame(quality_records).to_csv(OUTPUT_DIR / f'benchmark_quality_{tag}.csv', index=False)
pd.DataFrame(timing_records).to_csv(OUTPUT_DIR / f'benchmark_timing_{tag}.csv', index=False)
pd.DataFrame(candidate_records).to_csv(OUTPUT_DIR / f'benchmark_candidates_{tag}.csv', index=False)
pd.DataFrame(memory_records).to_csv(OUTPUT_DIR / f'benchmark_memory_{tag}.csv', index=False)
print('Saved benchmark CSVs.')


In [ ]:
summary = pd.DataFrame(quality_records).merge(
    pd.DataFrame(timing_records),
    on=['dataset', 'base_size', 'split_seed', 'hash_seed', 'method'],
)
print(summary.groupby(['dataset', 'base_size', 'method'])[['recall@10', 'median_latency_s', 'macro_f1']].mean())
